In [2]:
# -------------------------------------------------------------
# StreamWorks Churn Analysis — Beginner-Friendly End-to-End Code
# -------------------------------------------------------------
# What this script does:
# 1) Load & explore the data
# 2) Clean & prepare features (dates, engineered columns, encoding)
# 3) Optional feature engineering (simple, readable)
# 4) Statistical tests + simple charts
# 5) Predictive models:
#    - Logistic Regression for churn
#    - Linear Regression for average_watch_hours (continuous)
# 6) Prints simple, clear answers to the business questions
#
# Notes:
# - No seaborn is used; charts are matplotlib-only.
# - All plots are saved as PNG files in the working folder.
# - Code aims to be easy to read and modify.
# -------------------------------------------------------------

# === Imports ===
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, r2_score, mean_squared_error, mean_absolute_error
)

# === File path ===
# Put your CSV next to this script or change the path here
DATA_PATH = "streamworks_user_data.csv"

# === 1) Load & Explore ===
print("\n=== 1) LOAD & EXPLORE ===")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Update DATA_PATH to your CSV.")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print("\nData shape:", df.shape)

# .info() output captured as text
buf = io.StringIO()
df.info(buf=buf)
print("\nData .info():")
print(buf.getvalue())

print("\nFirst 5 rows:")
print(df.head())

print("\n.describe() (numerics):")
print(df.describe().T)

print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False))


=== 1) LOAD & EXPLORE ===

Data shape: (1500, 14)

Data .info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   user_id               1498 non-null   float64
 1   age                   1497 non-null   float64
 2   gender                1499 non-null   object 
 3   signup_date           1498 non-null   object 
 4   last_active_date      1498 non-null   object 
 5   country               1497 non-null   object 
 6   subscription_type     1497 non-null   object 
 7   average_watch_hours   1496 non-null   float64
 8   mobile_app_usage_pct  1498 non-null   float64
 9   complaints_raised     1497 non-null   float64
 10  received_promotions   1497 non-null   object 
 11  referred_by_friend    1497 non-null   object 
 12  is_churned            1499 non-null   float64
 13  monthly_fee           1355 non-null   float64
dtypes: flo

In [3]:
# === 2) Clean & Prepare the Data ===
print("\n=== 2) CLEAN & PREPARE ===")

# Parse dates (if present)
for col in ["signup_date", "last_active_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# tenure_days = days between signup and last_active_date
if all(c in df.columns for c in ["signup_date", "last_active_date"]):
    df["tenure_days"] = (df["last_active_date"] - df["signup_date"]).dt.days

# is_loyal = tenure_days > 180
if "tenure_days" in df.columns:
    df["is_loyal"] = (df["tenure_days"] > 180).astype(int)
else:
    df["is_loyal"] = 0  # fallback if dates are missing

# Helper: Yes/No → 1/0
def yesno_to_binary(series):
    return (
        series.astype(str)
              .str.strip()
              .str.lower()
              .map({"yes": 1, "no": 0})
    )

# Encode received_promotions / referred_by_friend if present
if "received_promotions" in df.columns:
    df["received_promotions_bin"] = yesno_to_binary(df["received_promotions"]).fillna(0).astype(int)
if "referred_by_friend" in df.columns:
    df["referred_by_friend_bin"] = yesno_to_binary(df["referred_by_friend"]).fillna(0).astype(int)

# Heavy mobile user (>= 60% mobile usage)
if "mobile_app_usage_pct" in df.columns:
    df["heavy_mobile_user"] = (df["mobile_app_usage_pct"] >= 60).astype(int)

# Watch-per-fee ratio (simple value feature)
if "average_watch_hours" in df.columns and "monthly_fee" in df.columns:
    denom = df["monthly_fee"].replace(0, np.nan)
    df["watch_per_fee_ratio"] = df["average_watch_hours"] / denom

# Fill missing values (simple, explainable approach)
# - numeric → median
# - categorical → mode
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

for c in numeric_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].median())

for c in categorical_cols:
    if df[c].isnull().any():
        mode_val = df[c].mode(dropna=True)
        if not mode_val.empty:
            df[c] = df[c].fillna(mode_val.iloc[0])
        else:
            df[c] = df[c].fillna("Unknown")

# One-hot encode low-cardinality categoricals (skip IDs)
low_card_cats = []
for c in categorical_cols:
    if c.lower() not in ["user_id", "id"] and df[c].nunique() <= 50:
        low_card_cats.append(c)

df_encoded = pd.get_dummies(df, columns=low_card_cats, drop_first=True)

# Drop datetime columns before modelling
datetime_cols = df_encoded.select_dtypes(include=["datetime64[ns]"]).columns.tolist()
df_encoded = df_encoded.drop(columns=datetime_cols, errors="ignore")

print("\nColumns after encoding & cleaning:", len(df_encoded.columns))


=== 2) CLEAN & PREPARE ===

Columns after encoding & cleaning: 24


/var/folders/ff/w7h1h0rj1nlf8p8qm9tz1h940000gn/T/ipykernel_95894/1891026980.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


In [4]:
# === 3) Feature Engineering (Optional extras already above) ===
# Already added: tenure_days, is_loyal, watch_per_fee_ratio, heavy_mobile_user, one-hot encodings
# You can add simple buckets if you want, e.g. age bands:
if "age" in df_encoded.columns:
    # creates a simple categorical, then one-hot if you like
    df_encoded["age_band"] = pd.cut(df_encoded["age"], bins=[0, 24, 34, 44, 54, 100], labels=[
        "<=24", "25-34", "35-44", "45-54", "55+"
    ])
    df_encoded = pd.get_dummies(df_encoded, columns=["age_band"], drop_first=True)

In [5]:
# === 4) Statistical Analysis & Charts ===
print("\n=== 4) STATS & CHARTS ===")
TARGET = "is_churned" if "is_churned" in df_encoded.columns else None

# Correlation heatmap for numeric features
num_for_corr = df_encoded.select_dtypes(include=[np.number])
if num_for_corr.shape[1] >= 2:
    corr = num_for_corr.corr()
    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Heatmap (numeric features)")
    plt.tight_layout()
    plt.savefig("corr_heatmap.png", dpi=200)
    plt.close()
    print("Saved: corr_heatmap.png")

# Chi-square tests: churn vs gender/promotions/referrals (if available)
def chi_square(df_in, cat_col, target_col):
    sub = df_in[[cat_col, target_col]].dropna()
    ct = pd.crosstab(sub[cat_col], sub[target_col])
    if ct.shape[0] >= 2 and ct.shape[1] >= 2:
        chi2, p, dof, expected = stats.chi2_contingency(ct)
        return p, ct
    return np.nan, ct

if TARGET:
    for col in ["gender", "received_promotions", "referred_by_friend"]:
        if col in df.columns:
            pval, table = chi_square(df, col, TARGET)
            print(f"\nChi-square: churn ~ {col} | p = {pval:.4f}")
            print(table)

# t-test: average_watch_hours for churned vs retained
if TARGET and "average_watch_hours" in df.columns:
    # Make sure target is numeric
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    churn = df.loc[df[TARGET] == 1, "average_watch_hours"].dropna()
    stay  = df.loc[df[TARGET] == 0, "average_watch_hours"].dropna()
    if len(churn) > 2 and len(stay) > 2:
        t_stat, p_val = stats.ttest_ind(churn, stay, equal_var=False)  # Welch's
        print("\nWelch t-test: average_watch_hours by churn")
        print(f"t = {t_stat:.3f}, p = {p_val:.4f} | mean(churn)={churn.mean():.2f}, mean(stay)={stay.mean():.2f}")

# Charts (saved as PNGs)
# Boxplot: watch hours by churn
if TARGET and "average_watch_hours" in df.columns:
    data = [
        df.loc[df[TARGET]==0, "average_watch_hours"].dropna(),
        df.loc[df[TARGET]==1, "average_watch_hours"].dropna()
    ]
    if all(len(d) > 0 for d in data):
        plt.figure(figsize=(6,4))
        plt.boxplot(data, labels=["Active (0)", "Churned (1)"])
        plt.title("Average Watch Hours by Churn Status")
        plt.ylabel("Average Watch Hours")
        plt.tight_layout()
        plt.savefig("box_watch_vs_churn.png", dpi=200)
        plt.close()
        print("Saved: box_watch_vs_churn.png")

# Bar: churn rate by heavy_mobile_user
if TARGET and "heavy_mobile_user" in df.columns:
    rates = df.groupby("heavy_mobile_user")[TARGET].mean()
    plt.figure(figsize=(6,4))
    x = np.arange(len(rates.index))
    plt.bar(x, rates.values)
    plt.xticks(x, [str(i) for i in rates.index])
    plt.title("Churn Rate by Heavy Mobile User")
    plt.xlabel("Heavy Mobile User (1 = yes)")
    plt.ylabel("Churn Rate")
    plt.tight_layout()
    plt.savefig("bar_churn_heavy_mobile.png", dpi=200)
    plt.close()
    print("Saved: bar_churn_heavy_mobile.png")

# Bar: churn rate by received promotions
if TARGET and "received_promotions_bin" in df.columns:
    rates = df.groupby("received_promotions_bin")[TARGET].mean()
    plt.figure(figsize=(6,4))
    x = np.arange(len(rates.index))
    plt.bar(x, rates.values)
    if len(rates) == 2:
        plt.xticks(x, ["No", "Yes"])
    else:
        plt.xticks(x, [str(i) for i in rates.index])
    plt.title("Churn Rate by Received Promotions")
    plt.xlabel("Received Promotions")
    plt.ylabel("Churn Rate")
    plt.tight_layout()
    plt.savefig("bar_churn_promotions.png", dpi=200)
    plt.close()
    print("Saved: bar_churn_promotions.png")

# Tenure histograms by churn
if TARGET and "tenure_days" in df.columns:
    mask_churn = df[TARGET] == 1
    a = df.loc[~mask_churn, "tenure_days"].dropna()
    b = df.loc[mask_churn, "tenure_days"].dropna()
    if len(a) > 0 and len(b) > 0:
        plt.figure(figsize=(6,4))
        plt.hist(a, bins=30, alpha=0.7, label="Active (0)")
        plt.hist(b, bins=30, alpha=0.7, label="Churned (1)")
        plt.title("Tenure Days Distribution by Churn")
        plt.xlabel("Tenure (days)")
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        plt.savefig("hist_tenure_by_churn.png", dpi=200)
        plt.close()
        print("Saved: hist_tenure_by_churn.png")


=== 4) STATS & CHARTS ===
Saved: corr_heatmap.png

Chi-square: churn ~ gender | p = 0.1071
is_churned  0.0  1.0
gender              
Female      375  136
Male        378  105
Other       396  110

Chi-square: churn ~ received_promotions | p = 0.1058
is_churned           0.0  1.0
received_promotions          
No                   573  193
Yes                  576  158

Chi-square: churn ~ referred_by_friend | p = 0.3818
is_churned          0.0  1.0
referred_by_friend          
No                  563  182
Yes                 586  169

Welch t-test: average_watch_hours by churn
t = -0.186, p = 0.8527 | mean(churn)=39.70, mean(stay)=39.97
Saved: box_watch_vs_churn.png
Saved: bar_churn_heavy_mobile.png
Saved: bar_churn_promotions.png


/var/folders/ff/w7h1h0rj1nlf8p8qm9tz1h940000gn/T/ipykernel_95894/2086143221.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=["Active (0)", "Churned (1)"])


Saved: hist_tenure_by_churn.png


In [5]:
# === 5) Predictive Modelling ===
print("\n=== 5) PREDICTIVE MODELLING ===")

# ------ Logistic Regression (Binary) ------
if TARGET and TARGET in df_encoded.columns:
    # Build X, y
    y_all = pd.to_numeric(df_encoded[TARGET], errors="coerce")
    X_all = df_encoded.drop(columns=[TARGET], errors="ignore")

    # Make sure X is numeric only
    X_all = X_all.select_dtypes(include=[np.number])

    # Drop rows with missing target
    mask = y_all.notna()
    X_all = X_all.loc[mask]
    y_all = y_all.loc[mask].astype(int)

    if X_all.shape[1] >= 2 and len(np.unique(y_all)) >= 2:
        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
        )

        # Scale numeric features (all are numeric now)
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        # Fit logistic regression
        logreg = LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")
        logreg.fit(X_train_sc, y_train)

        # Predict
        y_pred = logreg.predict(X_test_sc)
        y_prob = logreg.predict_proba(X_test_sc)[:, 1]

        # Metrics
        cm = confusion_matrix(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_prob)

        print("\nLogistic Regression Results")
        print("Confusion Matrix:\n", cm)
        print(f"Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f} | ROC-AUC: {auc:.3f}")

        # ROC Curve
        fpr, tpr, th = roc_curve(y_test, y_prob)
        plt.figure(figsize=(6,4))
        plt.plot(fpr, tpr)
        plt.plot([0,1],[0,1], linestyle="--")
        plt.title("ROC Curve — Logistic Regression")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.tight_layout()
        plt.savefig("roc_logreg.png", dpi=200)
        plt.close()
        print("Saved: roc_logreg.png")

        # Coefficients (top by absolute value)
        coef_df = pd.DataFrame({
            "feature": X_all.columns,
            "coef": logreg.coef_[0]
        }).sort_values("coef", key=lambda s: s.abs(), ascending=False)
        print("\nTop 10 Logistic Coefficients (by |coef|):")
        print(coef_df.head(10))
    else:
        print("Not enough features/classes to train the logistic model.")
else:
    print("No 'is_churned' target found — skipping logistic regression.")

# ------ Linear Regression (Continuous) ------
# We'll predict average_watch_hours if present
if "average_watch_hours" in df_encoded.columns:
    y_reg = pd.to_numeric(df_encoded["average_watch_hours"], errors="coerce")
    X_reg = df_encoded.drop(columns=["average_watch_hours"], errors="ignore")
    # Drop target if still in
    if TARGET in X_reg.columns:
        X_reg = X_reg.drop(columns=[TARGET], errors="ignore")

    # Only numeric columns
    X_reg = X_reg.select_dtypes(include=[np.number])

    # Drop rows with missing y
    mask_r = y_reg.notna()
    X_reg = X_reg.loc[mask_r]
    y_reg = y_reg.loc[mask_r]

    if X_reg.shape[1] >= 2:
        Xr_train, Xr_test, yr_train, yr_test = train_test_split(
            X_reg, y_reg, test_size=0.2, random_state=42
        )

        scaler_r = StandardScaler()
        Xr_train_sc = scaler_r.fit_transform(Xr_train)
        Xr_test_sc  = scaler_r.transform(Xr_test)

        linreg = LinearRegression()
        linreg.fit(Xr_train_sc, yr_train)
        yr_pred = linreg.predict(Xr_test_sc)

        r2 = r2_score(yr_test, yr_pred)
        rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))
        mae = mean_absolute_error(yr_test, yr_pred)

        print("\nLinear Regression Results (predicting average_watch_hours)")
        print(f"R2: {r2:.3f} | RMSE: {rmse:.3f} | MAE: {mae:.3f}")

        # Residual plot
        residuals = yr_test - yr_pred
        plt.figure(figsize=(6,4))
        plt.scatter(yr_pred, residuals, s=12)
        plt.axhline(0, linestyle="--")
        plt.title("Residuals vs Fitted — Linear Regression")
        plt.xlabel("Fitted Values")
        plt.ylabel("Residuals")
        plt.tight_layout()
        plt.savefig("residuals_linear.png", dpi=200)
        plt.close()
        print("Saved: residuals_linear.png")

        lin_coefs = pd.DataFrame({
            "feature": X_reg.columns,
            "coef": linreg.coef_
        }).sort_values("coef", key=lambda s: s.abs(), ascending=False)
        print("\nTop 10 Linear Coefficients (by |coef|):")
        print(lin_coefs.head(10))
    else:
        print("Not enough numeric features to train the linear model.")
else:
    print("No 'average_watch_hours' column found — skipping linear regression.")


=== 5) PREDICTIVE MODELLING ===

Logistic Regression Results
Confusion Matrix:
 [[120 110]
 [ 41  29]]
Precision: 0.209 | Recall: 0.414 | F1: 0.278 | ROC-AUC: 0.450
Saved: roc_logreg.png

Top 10 Logistic Coefficients (by |coef|):
                    feature      coef
11      watch_per_fee_ratio -0.284089
2       average_watch_hours  0.238725
5               monthly_fee -0.190680
8   received_promotions_bin -0.189752
7                  is_loyal  0.147672
6               tenure_days -0.066289
3      mobile_app_usage_pct  0.046494
9    referred_by_friend_bin -0.045592
4         complaints_raised -0.042446
10        heavy_mobile_user -0.031744

Linear Regression Results (predicting average_watch_hours)
R2: 0.809 | RMSE: 10.080 | MAE: 7.603
Saved: residuals_linear.png

Top 10 Linear Coefficients (by |coef|):
                    feature       coef
10      watch_per_fee_ratio  23.039514
4               monthly_fee  10.872931
5               tenure_days  -1.296760
6                  is_loyal 

In [6]:
# === 6) Business Questions (simple, printed answers) ===
print("\n=== 6) BUSINESS QUESTIONS ===")
def pct(x): 
    return "NA" if pd.isna(x) else f"{x:.3f}"

# 1. Do users who receive promotions churn less?
if TARGET and "received_promotions_bin" in df.columns:
    rates = df.groupby("received_promotions_bin")[TARGET].mean()
    ans1 = f"churn(Yes)={pct(rates.get(1, np.nan))} vs churn(No)={pct(rates.get(0, np.nan))}"
    print("1) Promotions vs churn:", ans1)
else:
    print("1) Not enough data to compare promotions vs churn.")

# 2. Does watch time impact churn likelihood?
if TARGET and "average_watch_hours" in df.columns:
    # We already did t-test above; summarize direction quickly
    churn_mean = df.loc[df[TARGET]==1, "average_watch_hours"].mean()
    stay_mean  = df.loc[df[TARGET]==0, "average_watch_hours"].mean()
    direction = "lower" if churn_mean < stay_mean else "higher"
    print(f"2) Watch time & churn: churned users have {direction} watch hours "
          f"(means: churned={pct(churn_mean)}, active={pct(stay_mean)}).")
else:
    print("2) Not enough data for watch-time comparison.")

# 3. Are mobile dominant users more likely to cancel?
if TARGET and "heavy_mobile_user" in df.columns:
    m_rates = df.groupby("heavy_mobile_user")[TARGET].mean()
    print("3) Churn by heavy-mobile:", {k: float(v) for k, v in m_rates.to_dict().items()})
else:
    print("3) Not enough data to compare heavy-mobile vs churn.")

# 4. Top 3 features influencing churn (logistic coefs)
if TARGET and os.path.exists("roc_logreg.png"):
    # We already printed coefficient table; just signal where to look
    print("4) Top churn predictors: see 'Top 10 Logistic Coefficients' printed above (take top 3 by |coef|).")
else:
    print("4) Logistic model not available → cannot list top predictors.")

# 5. Which customer segments should be prioritised?
print("5) Prioritise segments with: low watch time, recent complaints, no promotions, and any country/subscription dummies that show higher churn in your data.")

# 6. What factors affect watch time (linear regression insight)?
if "average_watch_hours" in df_encoded.columns:
    print("6) Watch-time drivers: see 'Top 10 Linear Coefficients' above (take top 3 by |coef|).")
else:
    print("6) Linear model not available → cannot list watch-time drivers.")

print("\nDone. Charts saved as PNGs in the working folder.")


=== 6) BUSINESS QUESTIONS ===
1) Promotions vs churn: churn(Yes)=0.215 vs churn(No)=0.252
2) Watch time & churn: churned users have lower watch hours (means: churned=39.704, active=39.966).
3) Churn by heavy-mobile: {0: 0.22502870264064295, 1: 0.246422893481717}
4) Top churn predictors: see 'Top 10 Logistic Coefficients' printed above (take top 3 by |coef|).
5) Prioritise segments with: low watch time, recent complaints, no promotions, and any country/subscription dummies that show higher churn in your data.
6) Watch-time drivers: see 'Top 10 Linear Coefficients' above (take top 3 by |coef|).

Done. Charts saved as PNGs in the working folder.
